In [5]:
from embedder import Embedder

embed = Embedder()
q="How does approximate nearest neighbor search work?"
v_q=embed.encode(q)
v_q[0]  #0.02

np.float64(-0.02058203437252893)

In [29]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
for doc in documents:
    if ( doc.get("filename")=="02-vector-search/lessons/07-sqlitesearch-vector.md"):
        e=embed.encode(doc["content"])
        print(e.dot(v_q))  #Q2: 0.37

0.36107027225589694


In [59]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np
X=[]

chunks = chunk_documents(documents, size=2000, step=1000)
chunk_texts = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(chunk_texts)
scores =X.dot(v_q)
idx = np.argmax(scores)
chunks[idx]["filename"] "Q3 :'02-vector-search/lessons/07-sqlitesearch-vector.md'

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [106]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)
#q="What metric do we use to evaluate a search engine?"
q="How do I store vectors in PostgreSQL?"
#q="How do I give the model access to tools?"
v_a=embed.encode(q)
vector_results=vindex.search(v_a,num_results=5)
print(vector_results[0]["filename"])  # Q4: 04-evaluation/lessons/05-search-metrics.md
#[ res["filename"] for res in vector_results ]

02-vector-search/lessons/08-pgvector.md


In [107]:
from minsearch import Index 
index= Index(text_fields=["content"])
index.fit(chunks)
q="How do I store vectors in PostgreSQL?"
#q="How do I give the model access to tools?"
text_results=index.search(q,num_results=5)
#print(text_results[0]["filename"])    #01-agentic-rag/lessons/05-search.md
[ res["filename"] for res in text_results ] #Q5 ;02-vector-search/lessons/08-pgvector.md

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [92]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

results = rrf([vector_results, text_results])
results[0]["filename"]  #Q6 :'01-agentic-rag/lessons/13-function-calling.md'

'01-agentic-rag/lessons/13-function-calling.md'